# Results and Figures

Purpose: load benchmark outputs, generate essential figures, and write the concise benchmark report.

Inputs: `results/tables/{output_prefix}__benchmark_results.csv`, metric summary, and run summary.

Outputs: benchmark figures, figure index, and Markdown report.

Matching scripts: `scripts/generate_figures.py` and `scripts/write_report.py`.


In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "rarecell").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from rarecell.config import FIGURES_DIR, LOGS_DIR, REPORTS_DIR, TABLES_DIR
from rarecell.plotting import save_all_standard_plots
from rarecell.reporting import write_benchmark_report


In [ ]:
run_summary_path = sorted(LOGS_DIR.glob("*__run_summary.json"), key=lambda p: p.stat().st_mtime)[-1]
run_summary = json.loads(run_summary_path.read_text())
output_prefix = run_summary["output_prefix"]
results_path = TABLES_DIR / f"{output_prefix}__benchmark_results.csv"
metric_summary_path = TABLES_DIR / f"{output_prefix}__metric_summary.csv"
results = pd.read_csv(results_path)
metric_summary = pd.read_csv(metric_summary_path)


In [ ]:
figure_paths = save_all_standard_plots(results, output_prefix, figures_dir=FIGURES_DIR, tables_dir=TABLES_DIR)
figure_index = pd.read_csv(TABLES_DIR / f"{output_prefix}__figure_index.csv")
run_summary["output_files"] = list(
    dict.fromkeys(run_summary.get("output_files", []) + [str(p.relative_to(PROJECT_ROOT)) for p in figure_paths]))
report_path = write_benchmark_report(
    REPORTS_DIR / f"{output_prefix}__benchmark_report.md",
    run_summary,
    metric_summary,
    figure_index,
)
report_path
